# Adding Conversions to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Conversions components**. Conversions components represent components that **converts one or several commodities into other commodities of the Energy System  Model**. They can be seen as black boxes, using the inputs commodities to create the output commodities. We focus in this notebook on the most essential parameters required to define and understand a Conversion component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sources include:

- a co-generation power plant using natural gas to produce electricity and heat 
- an electrolyzer using electricity to produce hydrogen
- a chemical plant using hydrogen, CO2, heat and electricity to produce methanol


## Add Conversions

## Load ESM

We first load the ESM from the [previous notebook](../_02_add_component/_2_add_sink.ipynb).

In [1]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

### Combined Cycle gas turbine plant

We can now add combined cycle gas turbine plant as a conversion component. Below you can find a more detailed explanation of the parameters used here.

In [2]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "naturalGas": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

## Save the Energy System Model

In [3]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission_conversion.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.5535 sec)


## General Structure of a Conversion Instance

The following code snippet shows the structure of the `Conversion` class and its arguments.

```python
Conversion(
    esM,                                    # defined in this notebook
    name,                                   # defined in this notebook
    physicalUnit,                           # defined in this notebook
    commodityConversionFactors,             # defined in this notebook
    hasCapacityVariable=True,               # defined in this notebook
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    linkedConversionCapacityID=None,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    operationRateMin=None,                  # defined in this notebook
    operationRateMax=None,                  # defined in this notebook
    operationRateFix=None,                  # defined in this notebook
    tsaWeight=1,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    QPcostScale=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    yearlyFullLoadHoursMin=None,
    yearlyFullLoadHoursMax=None,
    stockCommissioning=None,
    floorTechnicalLifetime=True,
    commissioningDependentCcf=False,
    emissionFactors=None,
    flowShares=None,
    pwlcfParameters=None,
    rampUpMax=None,
    rampDownMax=None,
    useTemporalCyclicConstraints=True,
)
```
In the following sections, we explain the most important arguments of a Conversion component.


## Required Arguments

### esM

`esM` is the energy system model to which the conversion is added.

### name

`name` is a string, which should describe the type of conversion which is added to the energy system model.

Examples:
- "nat_gas_power_plant"
- "PEM_electrolyzer"

### physicalUnit

`physicalUnit` defines the reference physical unit of the conversion component, to which maximum capacity limitations, cost parameters and the operation time series are all expressed.

Examples: 
- if `physicalUnit` = MW_{H2} for an electrolyzer, it means that its `capacityMax` of 10 MW is referred to hydrogen capacity and not electricity. 

### commodityConversionFactors

`commodityconversionfactor` specifies the conversion factors with which commodities are converted into each other with one unit of operation. The unit of operation were defined ealier in the [EnergySystemModel Initialization](../_01_initialize/_1_initialize_ESM.ipynb). The conversion factor related to this commodity is given as a float (constant), pandas.Series or pandas.DataFrame (time-variable). A negative value indicates that the commodity is consumed. A positive value indicates that the commodity is produced. Check unit consistency when specifying this parameter!

Examples: 
An electrolyzer converts, simply put, electricity into hydrogen with an electrical efficiency of 70%. The physicalUnit is given as GW_electric, the unit for the 'electricity' commodity is given in GW_electric and the 'hydrogen' commodity is given in GW_hydrogen_lowerHeatingValue -> the commodityConversionFactors are defined as {'electricity':-1,'hydrogen':0.7}.


## Optional Parameters

### hasCapacityVariable

`hasCapacityVariable` was alreadz defined previouslz, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### operationRateMax

`operationRateMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### capacityMax 
`capacityMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### investPerCapacity

`investPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### opexPerCapacity

`opexPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)


### opexPerOperation

`opexPerOperation` describes the cost for one unit of the operation. The cost which is directly proportional to the operation of the component is obtained by multiplying the opexPerOperation parameter with the annual sum of the operational time series of the components. The opexPerOperation can either be given as a float or a Pandas Series with location specific values. The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). |br| * the default value is 0 :type opexPerOperation: * Pandas Series with positive (>=0) entries. The indices of the series have to equal the in the energy system model specified locations. * a dictionary with investment periods as keys and one of the two options above as values.

### interestRate

`interestRate` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

### economicLifetime

`economicLifetime` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

## List of all parameters

Below, after executing the code cell, you will find the list of all parameters of a Conversion Component, along with their description, type, and default value.

In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.Conversion)

,Description,Type,Default
Argument,,,
esM,energy system model to which the component should be added. Used for unit checks.,EnergySystemModel instance from the FINE package,/
name,name of the component. Has to be unique (i.e. no other components with that name can already exist in the EnergySystemModel instance to which the component is added).,string,/
physicalUnit,"reference physical unit of the plants to which maximum capacity limitations, cost parameters and the operation time series refer to.",string,/
commodityConversionFactors,"conversion factors with which commodities are converted into each other with one unit of operation (dictionary). Each commodity which is converted in this component is indicated by a string in this dictionary. The conversion factor related to this commodity is given as a float (constant), pandas.Series or pandas.DataFrame (time-variable). A negative value indicates that the commodity is consumed. A positive value indicates that the commodity is produced. Check unit consistency when specifying this parameter! Examples: An electrolyzer converts, simply put, electricity into hydrogen with an electrical efficiency of 70%. The physicalUnit is given as GW_electric, the unit for the 'electricity' commodity is given in GW_electric and the 'hydrogen' commodity is given in GW_hydrogen_lowerHeatingValue -> the commodityConversionFactors are defined as {'electricity':-1,'hydrogen':0.7}. A fuel cell converts, simply put, hydrogen into electricity with an efficiency of 60%. The physicalUnit is given as GW_electric, the unit for the 'electricity' commodity is given in GW_electric and the 'hydrogen' commodity is given in GW_hydrogen_lowerHeatingValue -> the commodityConversionFactors are defined as {'electricity':1,'hydrogen':-1/0.6}. If a transformation pathway analysis is performed the conversion factors can also be varied over the transformation pathway. Therefore, two different options are available: 1. Variation with operation year (for example to incorporate weather changes for a heat pump). Example: {2020: {'electricity':-1,'heat':pd.Series(data=[2.5, 2.8, 2.5, ...])}, 2025: {'electricity':-1,'heat':pd.Series(data=[2.7, 2.4, 2.9, ...])}, ...} 2. Variation with commissioning and operation year (for example to incorporate efficiency changes dependent on the installation year). Please note that this implementation massively increases the complexity of the optimization problem. Example: {(2020, 2020): {'electricity':-1,'heat':pd.Series(data=[2.5, 2.8, 2.5, ...])}, (2020, 2025): {'electricity':-1,'heat':pd.Series(data=[2.7, 2.4, 2.9, ...])}, (2025, 2025): {'electricity':-1,'heat':pd.Series(data=[3.7, 3.4, 3.9, ...])}, ...} If a conversion component can decide between multiple in- or outputs which one to use (e.g. a chp plant) a flexible conversion component can be specified. This enables the component to substitute in- or output commodities within a commodity group. To allow this behavior an additional level needs to be specified: A CHP plant can decide between the production of heat or electricity (or a mix of both). When electricity is produced the conversion factor is 0.2 and for heat 0.5: {'gas': -1, 'out': {electricity: 0.2, heat: 0.5}}","dictionary, assigns commodities (string) to a conversion factors (float, pandas.Series or pandas.DataFrame) dictionary with investment periods as key and one of the first option as value dictionary with tuple of (commissioning year, investment period) as key and one of the first option above as value",/
hasCapacityVariable,"specifies if the component should be modeled with a capacity or not. Examples: An electrolyzer has a capacity given in GW_electric -> hasCapacityVariable is True. In the energy system, biogas can, from a model perspective, be converted into methane (and then used in conventional power plants which emit CO2) by getting CO2 from the environment. Thus, using biogas in conventional power plants is, from a balance perspective, CO2 free. This conversion is 